# EffectiveInference — финальный пайплайн (Colab, без API)

Замкнутая петля: **SFT (eval-loss + resume) → merge → генерация baseline+SFT → офлайн-оценка (локальный судья + авто-точность) → спот-чек**.

Идея: видеть прирост над стоковой моделью ДО сабмита, чтобы не жечь 4 попытки/сутки.
Перед запуском: Runtime → Change runtime type → GPU. Запускай ячейки строго по порядку.

In [ ]:
# 1. Зависимости
!pip install -q -U vllm==0.11.0 transformers==4.56.1 peft accelerate datasets bitsandbytes
# torchao 0.10 на Colab несовместим с peft (is_torchao_available() кидает ImportError).
!pip uninstall -y torchao

In [ ]:
# 2. Смонтировать Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Конфиг — ОБЯЗАТЕЛЬНО запусти ДО шагов ниже. Переменные кладём в os.environ,
# чтобы $VAR надёжно раскрывался в !-командах (subshell).
import os
PROJECT = '/content/drive/MyDrive/effinf_dev'      # папка с кодом и data/
os.environ['PROJECT']     = PROJECT
os.environ['BASE_MODEL']  = 'Qwen/Qwen3-1.7B'
os.environ['VARIANT']     = 'minimal'              # 'minimal' | 'rich'
os.environ['DRIVE']       = '/content/drive/MyDrive/effinf'  # адаптер (мелкий) — сюда
os.environ['MERGED']      = '/content/merged_minimal'        # merged-веса — на ЛОКАЛЬНЫЙ диск (быстро, без сбоев Drive)
os.environ['DTYPE']       = 'float16'              # T4. A100/L4 → 'bfloat16' или 'auto'
os.environ['JUDGE_MODEL'] = 'Qwen/Qwen2.5-3B-Instruct'  # L4/A100 → ...-7B-Instruct
os.makedirs(os.environ['DRIVE'], exist_ok=True)
%cd $PROJECT
!nvidia-smi --query-gpu=name,memory.total --format=csv && ls data/

## Шаг 1. SFT (полный сет, eval-loss по эпохам, чекпоинты на Drive)
Если сессия оборвётся — перезапусти эту ячейку, добавив `--resume`: докатит с последнего чекпоинта.
Для быстрой пробы добавь `--max_samples 3000 --epochs 1`.

In [ ]:
!python train_lora.py --variant "$VARIANT" \
    --train_jsonl "data/train_${VARIANT}.jsonl" \
    --eval_jsonl "data/eval.jsonl" \
    --output_dir "$DRIVE/lora_$VARIANT" \
    --base_model "$BASE_MODEL" \
    --max_samples 2000 --epochs 1 --max_len 640 \
    --batch 4 --grad_accum 4
# Лёгкий конфиг для T4 (~1.5ч). OOM → --batch 2 --grad_accum 8.
# Время режут только max_samples/epochs/max_len (батч не спасает).
# Обрыв сессии → перезапусти ячейку с --resume (докатит с чекпоинта).

## Шаг 2. Merge адаптера → safetensors

In [ ]:
!python merge_lora.py --base_model "$BASE_MODEL" \
    --adapter "$DRIVE/lora_$VARIANT" \
    --out "$MERGED"
# проверь, что веса реально записались (~3.4 ГБ model*.safetensors):
!ls -lh "$MERGED"

## Шаг 3. Генерация на held-out: стоковая (baseline) и дообученная (+ замер tok/s)
Две генерации, чтобы потом увидеть дельту. Каждая — свой `!python` (vLLM освобождает VRAM между ячейками).

In [ ]:
!python gen_candidates.py --model_dir "$BASE_MODEL" --variant "$VARIANT" \
    --out "data/cand_baseline.jsonl" --dtype "$DTYPE"

In [ ]:
!python gen_candidates.py --model_dir "$MERGED" --variant "$VARIANT" \
    --out "data/cand_$VARIANT.jsonl" --dtype "$DTYPE"

## Шаг 4. Офлайн-оценка — главный сигнал. Сравниваем baseline vs SFT.
Локальный судья (win-rate против эталона) + узкий sanity-check точности по \\boxed.

In [ ]:
print('--- BASELINE ---')
!python judge_local.py --candidates "data/cand_baseline.jsonl" --judge_model "$JUDGE_MODEL" --dtype "$DTYPE"
print('--- SFT ---')
!python judge_local.py --candidates "data/cand_$VARIANT.jsonl" --judge_model "$JUDGE_MODEL" --dtype "$DTYPE"

In [ ]:
!python auto_check.py --candidates "data/cand_baseline.jsonl"
!python auto_check.py --candidates "data/cand_$VARIANT.jsonl"

## Шаг 5. Спот-чек глазами

In [ ]:
!python peek.py --candidates "data/cand_$VARIANT.jsonl" --n 6

**Как читать (решение БЕЗ траты сабмита):**
- `win-rate` SFT должен быть заметно ВЫШЕ baseline (и желательно ближе к 0.5 — паритет с эталоном).
- `tok/s` со Шага 3 → влезаем ли в 15 мин (достоверно только на L4).
- `% упёршихся в max_tokens` → научилась ли модель ставить EOS.
- авто-точность по \\boxed — упираемся ли в способность модели (узкий, ~10% вопросов).

Если SFT уверенно бьёт baseline → забираем `merged_$VARIANT` в посылку (инструкция в чате)
и делаем ОДИН сабмит на подтверждение. Если нет — крутим данные/гиперпараметры офлайн.